In [ ]:
import lightkurve as lk
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from tqdm import tqdm
import os
import glob
import shutil
import time
from copy import deepcopy

from Kakapo.build_epsf import epsf_data_creation
from Kakapo.crappy_difference_image import Crappy_Difference_Imaging
from Kakapo.photometry import forced_photometry

import warnings

warnings.filterwarnings('ignore')

%matplotlib widget

In [ ]:
def _tpf_addition(tpf_info, tpf_input):
    
    if tpf_input.campaign is None:
        campaign = tpf_input.quarter
        mission = 'Kepler'
        print(f"Adding TPF {tpf_input.targetid} from {mission} quarter {campaign}")
    else:
        campaign = tpf_input.campaign
        mission = 'K2'
    
    # print(tpf_info)
    tpf_info.loc[len(tpf_info)] = [mission, campaign, tpf_input.targetid, tpf_input.ra, tpf_input.dec, 
                                   tpf_input.flux.value, tpf_input.flux_err.value, tpf_input.quality, 
                                   tpf_input.pos_corr1, tpf_input.pos_corr2, tpf_input.time]
    
    return tpf_info

def _check_tpf_type(tpf_input):
    
    tpf_info = pd.DataFrame(columns=['mission', 'campaign', 'targetid', 'ra', 'dec', 'flux', 'flux_err', 
                                     'quality', 'pos_corr1', 'pos_corr2', 'time'])
    
    if isinstance(tpf_input, lk.targetpixelfile.KeplerTargetPixelFile):
        tpf_info = _tpf_addition(tpf_info, tpf_input)
    
    elif isinstance(tpf_input, str):
        tpf_input = lk.open(tpf_input, quality_bitmask = 'none')
        tpf_info = _tpf_addition(tpf_info, tpf_input)
    
    elif isinstance(tpf_input, lk.collections.TargetPixelFileCollection):
        tpf_input_list = deepcopy(tpf_input)
        
        for i in range(len(tpf_input)):
            tpf_info = _tpf_addition(tpf_info, tpf_input_list[i])
            
    elif isinstance(tpf_input, list):
        print('LIST')
        tpf_input_list = deepcopy(tpf_input)
        
        for i in range(len(tpf_input)):
            if isinstance(tpf_input_list[i], lk.targetpixelfile.KeplerTargetPixelFile):
                tpf_info = _tpf_addition(tpf_info, tpf_input_list[i])
            elif isinstance(tpf_input_list[i], str):
                tpf_input_list[i] = lk.open(tpf_input_list[i], quality_bitmask = 'none')
                tpf_info = _tpf_addition(tpf_info, tpf_input_list[i])
            else:
                raise ValueError("tpf_input must be a string, a KeplerTargetPixelFile object, or a list of KeplerTargetPixelFile objects")
    else:
        raise ValueError("tpf_input must be a string, a KeplerTargetPixelFile object, or a list of KeplerTargetPixelFile objects")
    
    return tpf_info

In [ ]:
def access_tpfs():
    """
    """

    test_case = []

    lightkurve_file_folder = '/Users/zgl12/.lightkurve/cache/mastDownload/K2/'

    files = sorted(glob.glob(lightkurve_file_folder + '*/*.fits.gz'))

    for file in tqdm(files, desc='Reading TPFs'):
        tpf = lk.read(file, quality_bitmask = 'none')
        test_case.append(tpf)
        
    return test_case

In [ ]:
test_case = access_tpfs()

In [ ]:
for i in range(len(test_case)):
     if test_case[i].campaign == 3:
        print(i, test_case[i].targetid, test_case[i].campaign, test_case[i].ra, test_case[i].dec)

In [ ]:
epsf_data = epsf_data_creation(test_case, path = '/Users/zgl12/Modules/Kakapo/', overwrite = False, stop_cond = 3000, sampling = 1)

plt.figure()
plt.imshow(epsf_data[2:-2, 2:-2], origin = 'lower')
plt.show()

In [ ]:
tpf_info = _check_tpf_type(test_case)

In [ ]:
tpf_info

In [ ]:
for i in [4, 5, 6]:
    crappy_kea = Crappy_Difference_Imaging(tpf_info.loc[i], epsf_data[2:-2, 2:-2])
    
    np.save(f"crap_diff_image_t{tpf_info.loc[i].targetid}_c{tpf_info.loc[i].campaign}.npy", crappy_kea.difference_images)